In [1]:
import pandas as pd
import re

# Load raw data
df = pd.read_csv('canada_weather.csv')

def clean_temp(val):
    if pd.isna(val): return None
    # Replace unicode minus (−) with standard hyphen (-)
    val = str(val).replace('−', '-')
    # Extract the first float (Celsius)
    match = re.search(r"[-+]?\d*\.\d+|\d+", val)
    return float(match.group()) if match else None

def clean_elevation(val):
    if pd.isna(val): return 0.0
    # Strip 'm', commas, and take the part before the metric unit
    val = str(val).split('m')[0].replace(',', '')
    try: return float(val)
    except: return 0.0

def extract_lat(val):
    # Extract decimal latitude from the coordinate string
    match = re.search(r"([-+]?\d*\.\d+);", val)
    return float(match.group(1)) if match else None

# Apply cleaning
df['AnnualAvgLow_C'] = df['Annual(Avg. low °C (°F))'].apply(clean_temp)
df['Elevation_m'] = df['Elevation'].apply(clean_elevation)
df['Latitude'] = df['Location'].apply(extract_lat)

# Define target: Is_Cold_Zone (Annual Low < 0)
df['Is_Cold_Zone'] = (df['AnnualAvgLow_C'] < 0).astype(int)

# Export for SQL Import
df_cleaned = df[['Community', 'Elevation_m', 'Latitude', 'AnnualAvgLow_C', 'Is_Cold_Zone']]
df_cleaned.to_csv('cleaned_weather_data.csv', index=False)
print("Step 1 Complete: Cleaned CSV generated.")

Step 1 Complete: Cleaned CSV generated.


In [4]:
import pandas as pd
import pyodbc
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Connect to SQL
conn = pyodbc.connect("Driver={SQL Server};Server=Kelsey;Database=Lab6;Trusted_Connection=yes;")

# Pull Data
query = "SELECT * FROM cleaned_weather_data"
data = pd.read_sql(query, conn)

# Prepare Features (X) and Target (y)
X = data[['Elevation_m', 'Latitude']]
y = data['Is_Cold_Zone']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Model
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# Generate Predictions for the WHOLE dataset for auditing
data['Predicted_Zone'] = model.predict(X)
data['Is_Error'] = (data['Predicted_Zone'] != data['Is_Cold_Zone']).astype(int)

# Write results BACK to SQL for Phase 4
cursor = conn.cursor()
cursor.execute("CREATE TABLE Model_Audit_Results (Community VARCHAR(255), Actual INT, Predicted INT, Is_Error INT)")
for index, row in data.iterrows():
    cursor.execute("INSERT INTO Model_Audit_Results VALUES (?,?,?,?)", 
                   (row['Community'], row['Is_Cold_Zone'], row['Predicted_Zone'], row['Is_Error']))
conn.commit()
print("Step 3 Complete: Predictions saved to SQL.")

C:\Users\kelse\AppData\Local\Temp\ipykernel_11060\3948188744.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


Step 3 Complete: Predictions saved to SQL.
